# 02 — Qwen2.5-Coder 7B QLoRA: Code Snippet → Vulnerability Description

**Goal:** Fine-tune `Qwen2.5-Coder-7B-Instruct` with QLoRA on BigVul + CIRCL samples.  
The model's job is **not** CWE classification — that's RoBERTa's job.  
Its job is: given raw code, produce a structured NL description that RoBERTa can classify.

**Output format Qwen must learn:**
```
This function performs <operation> on <input> without <missing check>,
which may allow an attacker to <impact>.
```
Freeform explanations are not useful here — RoBERTa needs consistent phrasing.

**Output:** LoRA adapter pushed to HF Hub at `your-username/vuln-analyzer-qwen-lora`.  
Expected runtime on Kaggle T4: ~90–120 min.

## 0. Prerequisites

- Notebook 01 must be complete and checkpoint live on HF Hub
- Kaggle secret `HF_TOKEN` must be set (Add-ons → Secrets)
- Enable GPU: Settings → Accelerator → GPU T4 x1

In [1]:
%%capture
!pip install transformers datasets peft bitsandbytes accelerate huggingface_hub trl -q

## 1. Config — change only this cell

In [ ]:
from kaggle_secrets import UserSecretsClient
HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")

HF_USERNAME      = ""
HF_REPO_NAME     = "vuln-analyzer-qwen-lora"

BASE_MODEL       = "Qwen/Qwen2.5-Coder-7B-Instruct"

# QLoRA settings — tuned for T4 15GB
LORA_R           = 16
LORA_ALPHA       = 32
LORA_DROPOUT     = 0.05
LORA_TARGET      = ["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"]

MAX_LENGTH       = 1024   # code snippets can be long
BATCH_SIZE       = 2      # small batch — model is large
GRAD_ACCUM       = 8      # effective batch = 16
EPOCHS           = 3
LR               = 2e-4
WARMUP_RATIO     = 0.05
SEED             = 42

# How many BigVul samples to use — full dataset is ~188k but many are near-duplicates
# 5k gives good coverage without multi-hour training
MAX_BIGVUL       = 5000
MAX_CIRCL        = 2000   # CIRCL has ~39k — take a stratified sample

## 2. HF Hub login

In [3]:
from huggingface_hub import login
login(token=HF_TOKEN, add_to_git_credential=False)
print("Logged in to HF Hub")

Logged in to HF Hub


## 3. Load and prepare training data

We need `(code_snippet, cwe_id)` pairs to generate training examples.  
Sources: BigVul (C/C++) + CIRCL vulnerability dataset.

In [4]:
from datasets import load_dataset, Dataset, concatenate_datasets
import pandas as pd

# ── BigVul ────────────────────────────────────────────────────────────────────
# HF mirror: benjamindavid/bigvul — check column names on first run
print("Loading BigVul...")
bigvul_raw = load_dataset("bstee615/bigvul", split="train")
print("BigVul columns:", bigvul_raw.column_names)
print("BigVul sample:", bigvul_raw[0])

Loading BigVul...
BigVul columns: ['CVE ID', 'CVE Page', 'CWE ID', 'codeLink', 'commit_id', 'commit_message', 'func_after', 'func_before', 'lang', 'project', 'vul']
BigVul sample: {'CVE ID': 'CVE-2017-7586', 'CVE Page': 'https://www.cvedetails.com/cve/CVE-2017-7586/', 'CWE ID': 'CWE-119', 'codeLink': 'https://github.com/erikd/libsndfile/commit/708e996c87c5fae77b104ccfeb8f6db784c32074', 'commit_id': '708e996c87c5fae77b104ccfeb8f6db784c32074', 'commit_message': 'src/ : Move to a variable length header buffer\n\nPreviously, the `psf->header` buffer was a fixed length specified by\n`SF_HEADER_LEN` which was set to `12292`. This was problematic for\ntwo reasons; this value was un-necessarily large for the majority\nof files and too small for some others.\n\nNow the size of the header buffer starts at 256 bytes and grows as\nnecessary up to a maximum of 100k.', 'func_after': 'psf_get_date_str (char *str, int maxlen)\n{\ttime_t\t\tcurrent ;\n\tstruct tm\ttimedata, *tmptr ;\n\n\ttime (&current

In [5]:
# Adjust column names below if the print above shows different names
BIGVUL_CODE_COL  = "func_before"   # vulnerable function before patch
BIGVUL_CWE_COL   = "CWE ID"        # may be "cwe_id" — check above
BIGVUL_VULN_COL  = "vul"           # 1 = vulnerable, 0 = clean

MITRE_TOP_25 = [
    "CWE-787", "CWE-79",  "CWE-89",  "CWE-416", "CWE-78",
    "CWE-20",  "CWE-125", "CWE-22",  "CWE-352", "CWE-434",
    "CWE-862", "CWE-476", "CWE-287", "CWE-190", "CWE-502",
    "CWE-77",  "CWE-119", "CWE-798", "CWE-918", "CWE-306",
    "CWE-362", "CWE-269", "CWE-94",  "CWE-863", "CWE-276"
]

# Filter: vulnerable only, Top 25 CWEs, non-empty code
bigvul_df = bigvul_raw.to_pandas()
bigvul_df = bigvul_df[
    (bigvul_df[BIGVUL_VULN_COL] == 1) &
    (bigvul_df[BIGVUL_CWE_COL].isin(MITRE_TOP_25)) &
    (bigvul_df[BIGVUL_CODE_COL].notna()) &
    (bigvul_df[BIGVUL_CODE_COL].str.len() > 50)
].copy()

# Sample — stratify by CWE so no class dominates
bigvul_df = (
    bigvul_df.groupby(BIGVUL_CWE_COL, group_keys=False)
    .apply(lambda g: g.sample(min(len(g), MAX_BIGVUL // 25), random_state=SEED))
    .reset_index(drop=True)
)

print(f"BigVul samples after filter: {len(bigvul_df)}")
print(bigvul_df[BIGVUL_CWE_COL].value_counts())

BigVul samples after filter: 1681
CWE ID
CWE-119    200
CWE-125    200
CWE-190    200
CWE-20     200
CWE-416    200
CWE-362    200
CWE-476    170
CWE-787    155
CWE-79      41
CWE-22      27
CWE-269     27
CWE-287     20
CWE-78      11
CWE-77      10
CWE-94       8
CWE-862      4
CWE-89       4
CWE-918      2
CWE-502      1
CWE-352      1
Name: count, dtype: int64


/tmp/ipykernel_779/2149154629.py:26: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(min(len(g), MAX_BIGVUL // 25), random_state=SEED))


In [6]:
# ── CIRCL ─────────────────────────────────────────────────────────────────────
# CIRCL dataset has patch diffs — we use the 'before' state (vulnerable code)
# Dataset: CIRCL/vulnerability-cwe-patch
print("Loading CIRCL...")
try:
    circl_raw = None  # CIRCL patch format needs preprocessing — skip for v0.1
    print("CIRCL columns:", circl_raw.column_names)
    print("CIRCL sample:", circl_raw[0])
except Exception as e:
    print(f"CIRCL load failed: {e}")
    print("Continuing with BigVul only — add CIRCL manually if needed")
    circl_raw = None

Loading CIRCL...
CIRCL load failed: 'NoneType' object has no attribute 'column_names'
Continuing with BigVul only — add CIRCL manually if needed


In [7]:
# Build unified DataFrame: columns = ['code', 'cwe_id']
unified_rows = []

for _, row in bigvul_df.iterrows():
    unified_rows.append({
        "code":   str(row[BIGVUL_CODE_COL]),
        "cwe_id": str(row[BIGVUL_CWE_COL])
    })

if circl_raw is not None:
    # Adjust column names based on the print above
    CIRCL_CODE_COL = "vulnerable_code"   # check and update
    CIRCL_CWE_COL  = "cwe_id"            # check and update

    circl_df = circl_raw.to_pandas()
    circl_df = circl_df[
        (circl_df[CIRCL_CWE_COL].isin(MITRE_TOP_25)) &
        (circl_df[CIRCL_CODE_COL].notna()) &
        (circl_df[CIRCL_CODE_COL].str.len() > 50)
    ].sample(min(len(circl_df), MAX_CIRCL), random_state=SEED)

    for _, row in circl_df.iterrows():
        unified_rows.append({
            "code":   str(row[CIRCL_CODE_COL]),
            "cwe_id": str(row[CIRCL_CWE_COL])
        })

unified_df = pd.DataFrame(unified_rows)
print(f"\nTotal training pairs: {len(unified_df)}")
print(unified_df["cwe_id"].value_counts())


Total training pairs: 1681
cwe_id
CWE-119    200
CWE-125    200
CWE-190    200
CWE-20     200
CWE-416    200
CWE-362    200
CWE-476    170
CWE-787    155
CWE-79      41
CWE-22      27
CWE-269     27
CWE-287     20
CWE-78      11
CWE-77      10
CWE-94       8
CWE-862      4
CWE-89       4
CWE-918      2
CWE-502      1
CWE-352      1
Name: count, dtype: int64


## 4. Build instruction dataset

Each training example is an instruction-following pair:  
- **Instruction:** "Analyze this code and describe the vulnerability in one structured sentence."  
- **Output:** The structured description Qwen should learn to produce.

We generate outputs using the CWE ID + a template — no LLM needed for this step.

In [8]:
# CWE → description templates
# These give Qwen a target output format it can learn to generalize from
CWE_TEMPLATES = {
    "CWE-787": "This function writes to memory beyond the bounds of the allocated buffer without checking the size of the input, which may allow an attacker to corrupt memory or execute arbitrary code.",
    "CWE-79":  "This function reflects user-supplied input into the HTTP response without encoding or sanitization, which may allow an attacker to inject and execute malicious scripts in the victim's browser.",
    "CWE-89":  "This function constructs a SQL query by concatenating user-controlled input without parameterization, which may allow an attacker to inject arbitrary SQL commands and access or modify the database.",
    "CWE-416": "This function uses a pointer to memory after it has been freed, which may allow an attacker to corrupt memory or achieve arbitrary code execution through a use-after-free condition.",
    "CWE-78":  "This function passes user-controlled input to a system command without sanitization, which may allow an attacker to inject and execute arbitrary OS commands.",
    "CWE-20":  "This function processes external input without validating its type, length, or format, which may allow an attacker to supply unexpected values that trigger incorrect behavior.",
    "CWE-125": "This function reads memory beyond the end of the allocated buffer without bounds checking, which may allow an attacker to read sensitive data from adjacent memory.",
    "CWE-22":  "This function constructs a file path using user-supplied input without normalizing or validating it, which may allow an attacker to traverse the directory structure and access files outside the intended directory.",
    "CWE-352": "This function processes state-changing requests without verifying that they originate from the authenticated user, which may allow an attacker to perform actions on behalf of the victim through cross-site request forgery.",
    "CWE-434": "This function accepts file uploads without validating the file type or content, which may allow an attacker to upload malicious files that are later executed on the server.",
    "CWE-862": "This function performs a sensitive operation without verifying that the caller has the required authorization, which may allow an unauthorized user to access restricted functionality.",
    "CWE-476": "This function dereferences a pointer without checking whether it is null, which may allow an attacker to trigger a null pointer dereference and crash the application.",
    "CWE-287": "This function grants access without properly verifying the identity of the requester, which may allow an attacker to bypass authentication and gain unauthorized access.",
    "CWE-190": "This function performs arithmetic on an integer value without checking for overflow, which may allow an attacker to cause the result to wrap around to an unexpected value and trigger incorrect behavior.",
    "CWE-502": "This function deserializes data from an untrusted source without validation, which may allow an attacker to supply crafted serialized objects that execute arbitrary code during deserialization.",
    "CWE-77":  "This function passes user-controlled input to a command interpreter without sanitization, which may allow an attacker to inject additional commands.",
    "CWE-119": "This function performs operations on a memory buffer without verifying that the operation stays within the bounds of the buffer, which may allow an attacker to read or write out-of-bounds memory.",
    "CWE-798": "This function contains a hardcoded credential embedded in the source code, which may allow an attacker who obtains the code to authenticate as a privileged user.",
    "CWE-918": "This function makes server-side HTTP requests to a URL supplied by the user without restricting the target, which may allow an attacker to proxy requests to internal services.",
    "CWE-306": "This function performs a sensitive operation without requiring authentication, which may allow an unauthenticated attacker to access restricted functionality.",
    "CWE-362": "This function accesses a shared resource from multiple execution contexts without proper synchronization, which may allow a race condition that leads to incorrect behavior or privilege escalation.",
    "CWE-269": "This function grants excessive privileges to a process or user without restricting them to the minimum required, which may allow an attacker to escalate privileges.",
    "CWE-94":  "This function incorporates user-controlled input into code that is later executed, which may allow an attacker to inject and run arbitrary code.",
    "CWE-863": "This function verifies that a user is authenticated but does not check whether they are authorized to perform the requested operation, which may allow a low-privilege user to access resources belonging to other users.",
    "CWE-276": "This function or resource is configured with permissions that are broader than necessary, which may allow unintended users to read, write, or execute it.",
}

SYSTEM_PROMPT = """You are a security analyst. Given a code snippet, produce exactly one structured sentence describing the vulnerability it contains.

Format: "This function performs <operation> on <input> without <missing check>, which may allow an attacker to <impact>."

Be specific about the operation and the missing check. Do not add any other text."""

def build_chat_example(code: str, cwe_id: str) -> dict:
    """Build a single instruction-following example in Qwen chat format."""
    # Truncate very long functions to avoid blowing context
    code_truncated = code[:3000] if len(code) > 3000 else code

    messages = [
        {"role": "system",    "content": SYSTEM_PROMPT},
        {"role": "user",      "content": f"Analyze this code:\n\n```\n{code_truncated}\n```"},
        {"role": "assistant", "content": CWE_TEMPLATES.get(cwe_id, f"This function contains a {cwe_id} vulnerability that may allow an attacker to cause harm.")}
    ]
    return {"messages": messages, "cwe_id": cwe_id}

examples = [build_chat_example(row["code"], row["cwe_id"])
            for _, row in unified_df.iterrows()]

dataset = Dataset.from_list(examples)
split   = dataset.train_test_split(test_size=0.05, seed=SEED)
train_ds, eval_ds = split["train"], split["test"]
print(f"Train: {len(train_ds)} | Eval: {len(eval_ds)}")

Train: 1596 | Eval: 85


## 5. Load model in 4-bit (QLoRA)

In [9]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True   # saves ~0.4 bits/param extra
)

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"    # required for causal LM training

print("Loading model in 4-bit...")
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)
model.config.use_cache = False   # required for gradient checkpointing

print(f"Model loaded. GPU mem: {torch.cuda.memory_allocated()/1e9:.1f} GB")

Loading tokenizer...
Loading model in 4-bit...


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Model loaded. GPU mem: 1.5 GB


In [10]:
from peft import prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGET,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
# Expect: ~0.7% of params trainable — that's correct for QLoRA r=16

trainable params: 40,370,176 || all params: 7,655,986,688 || trainable%: 0.5273


## 6. Tokenize with chat template

In [11]:
def tokenize_chat(example):
    """Apply Qwen chat template and tokenize. Mask prompt tokens in labels."""
    # Apply Qwen's built-in chat template
    full_text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False
    )

    # Tokenize full conversation
    tokenized = tokenizer(
        full_text,
        truncation=True,
        max_length=MAX_LENGTH,
        padding="max_length"
    )

    # Build labels: -100 for prompt tokens (we only train on assistant output)
    # Find where assistant response starts
    prompt_messages = example["messages"][:-1]  # everything except assistant turn
    prompt_text = tokenizer.apply_chat_template(
        prompt_messages,
        tokenize=False,
        add_generation_prompt=True
    )
    prompt_len = len(tokenizer(prompt_text, truncation=True,
                               max_length=MAX_LENGTH)["input_ids"])

    labels = tokenized["input_ids"].copy()
    labels[:prompt_len] = [-100] * prompt_len   # mask prompt
    # Also mask padding
    labels = [-100 if t == tokenizer.pad_token_id else l
              for t, l in zip(tokenized["input_ids"], labels)]
    tokenized["labels"] = labels

    return tokenized

# Verify on one example before running the full map
sample = tokenize_chat(train_ds[0])
non_masked = sum(1 for l in sample["labels"] if l != -100)
print(f"Non-masked label tokens (assistant output): {non_masked}")
# Should be 30–80 tokens — the structured description is short

Non-masked label tokens (assistant output): 31


In [12]:
tokenized_train = train_ds.map(tokenize_chat, remove_columns=train_ds.column_names)
tokenized_eval  = eval_ds.map(tokenize_chat,  remove_columns=eval_ds.column_names)
tokenized_train.set_format("torch")
tokenized_eval.set_format("torch")
print("Tokenization done")

Map:   0%|          | 0/1596 [00:00<?, ? examples/s]

Map:   0%|          | 0/85 [00:00<?, ? examples/s]

Tokenization done


## 7. Train

In [13]:
from transformers import TrainingArguments, Trainer, DataCollatorForSeq2Seq

output_dir = f"/kaggle/working/{HF_REPO_NAME}"

training_args = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    warmup_ratio=WARMUP_RATIO,
    weight_decay=0.01,
    fp16=True,
    gradient_checkpointing=False,     # cuts memory ~30% at the cost of speed
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    logging_steps=25,
    seed=SEED,
    report_to="none",
    optim="paged_adamw_8bit"         # bitsandbytes paged optimizer — saves memory
)

from transformers import default_data_collator
collator = default_data_collator

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    data_collator=collator,
)

print(f"Training on {len(tokenized_train):,} examples")
print(f"Effective batch size: {BATCH_SIZE * GRAD_ACCUM}")
print(f"GPU mem before training: {torch.cuda.memory_allocated()/1e9:.1f} GB")
trainer.train()

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Training on 1,596 examples
Effective batch size: 16
GPU mem before training: 2.6 GB


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: The AccumulateGrad node's stream does not match the stream of the node that produced the incoming gradient. This may incur unnecessary synchronization and break CUDA graph capture if the AccumulateGrad node's stream is the default stream. This mismatch is caused by an AccumulateGrad node created prior to the current iteration being kept alive. This can happen if the autograd graph is still being kept alive by tensors such a

Epoch,Training Loss,Validation Loss
1,0.084461,0.067455
2,0.054916,0.053730
3,0.037718,0.045841


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


TrainOutput(global_step=300, training_loss=0.11531793137391408, metrics={'train_runtime': 19882.446, 'train_samples_per_second': 0.241, 'train_steps_per_second': 0.015, 'total_flos': 2.0918732897805926e+17, 'train_loss': 0.11531793137391408, 'epoch': 3.0})

## 8. Save adapter and push to HF Hub

We push **only the LoRA adapter** (~200MB), not the full 7B weights.  
At runtime, `pipeline/code_analyzer.py` loads the base model + merges the adapter.

In [14]:
HF_REPO_ID = f"{HF_USERNAME}/{HF_REPO_NAME}"

# Save adapter locally first
model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)

# Push adapter to HF Hub
model.push_to_hub(HF_REPO_ID, token=HF_TOKEN)
tokenizer.push_to_hub(HF_REPO_ID, token=HF_TOKEN)

print(f"\nAdapter live at: https://huggingface.co/{HF_REPO_ID}")
print("Note: this repo contains only the LoRA adapter, not the full model.")
print(f"At runtime, load with: PeftModel.from_pretrained(base_model, '{HF_REPO_ID}')")

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            


Adapter live at: https://huggingface.co/martynattakit/vuln-analyzer-qwen-lora
Note: this repo contains only the LoRA adapter, not the full model.
At runtime, load with: PeftModel.from_pretrained(base_model, 'martynattakit/vuln-analyzer-qwen-lora')


## 9. Sanity check — generate descriptions for test snippets

In [15]:
from peft import PeftModel

def generate_description(code: str, max_new_tokens: int = 120) -> str:
    """Run inference: code snippet → structured vulnerability description."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": f"Analyze this code:\n\n```\n{code}\n```"}
    ]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,           # greedy — consistent output for eval
            temperature=1.0,
            pad_token_id=tokenizer.eos_token_id
        )

    # Decode only the new tokens (skip the prompt)
    new_tokens = output[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


# Test cases — known vulnerable patterns
test_snippets = [
    # SQL injection (CWE-89)
    ("""
def get_user(username):
    query = "SELECT * FROM users WHERE name = '" + username + "'"
    return db.execute(query)
""", "Expected: CWE-89 SQL injection"),

    # Buffer overflow (CWE-787)
    ("""
void copy_input(char *dst, char *src) {
    strcpy(dst, src);
}
""", "Expected: CWE-787 out-of-bounds write"),

    # Command injection (CWE-78)
    ("""
def run_ping(host):
    os.system("ping -c 1 " + host)
""", "Expected: CWE-78 OS command injection"),
]

for code, label in test_snippets:
    desc = generate_description(code)
    print(f"{label}")
    print(f"Output: {desc}")
    print()

The following generation flags are not valid and may be ignored: ['top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Expected: CWE-89 SQL injection
Output: This function constructs a SQL query by concatenating user-controlled input without parameterization, which may allow an attacker to inject arbitrary SQL commands and access or modify the database.

Expected: CWE-787 out-of-bounds write
Output: This function performs operations on a memory buffer without verifying that the operation stays within the bounds of the buffer, which may allow an attacker to read or write out-of-bounds memory.

Expected: CWE-78 OS command injection
Output: This function passes user-controlled input to a system command without sanitization, which may allow an attacker to inject and execute arbitrary OS commands.



## 10. End-to-end test: code → Qwen → RoBERTa

This tests the actual pipeline handoff — Qwen's description goes into RoBERTa for CWE classification.  
If this works, `pipeline/code_analyzer.py` + `pipeline/classifier.py` are ready to wire together.

In [16]:
from transformers import pipeline as hf_pipeline

# Load the RoBERTa classifier from notebook 01
ROBERTA_REPO = f"{HF_USERNAME}/vuln-classifier-roberta"
roberta_clf = hf_pipeline(
    "text-classification",
    model=ROBERTA_REPO,
    token=HF_TOKEN,
    top_k=3
)

print("Full pipeline: code → Qwen description → RoBERTa classification\n")

for code, label in test_snippets:
    description = generate_description(code)
    classifications = roberta_clf(description)

    print(f"{label}")
    print(f"  Qwen:    {description}")
    print(f"  RoBERTa top-3:")
    for c in classifications[0]:
        print(f"    {c['label']}: {c['score']:.3f}")
    print()

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/359 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Full pipeline: code → Qwen description → RoBERTa classification

Expected: CWE-89 SQL injection
  Qwen:    This function constructs a SQL query by concatenating user-controlled input without parameterization, which may allow an attacker to inject arbitrary SQL commands and access or modify the database.
  RoBERTa top-3:
    CWE-89: 1.000
    CWE-306: 0.000
    CWE-287: 0.000

Expected: CWE-787 out-of-bounds write
  Qwen:    This function performs operations on a memory buffer without verifying that the operation stays within the bounds of the buffer, which may allow an attacker to read or write out-of-bounds memory.
  RoBERTa top-3:
    CWE-119: 0.773
    CWE-787: 0.196
    CWE-125: 0.025

Expected: CWE-78 OS command injection
  Qwen:    This function passes user-controlled input to a system command without sanitization, which may allow an attacker to inject and execute arbitrary OS commands.
  RoBERTa top-3:
    CWE-78: 0.994
    CWE-77: 0.004
    CWE-94: 0.000



---
## What to record after this run

| Item | Value |
|------|-------|
| Final eval loss | _(fill in)_ |
| SQL injection test — RoBERTa top-1 | _(should be CWE-89)_ |
| Buffer overflow test — RoBERTa top-1 | _(should be CWE-787)_ |
| Command injection test — RoBERTa top-1 | _(should be CWE-78)_ |
| Adapter HF URL | `https://huggingface.co/your-username/vuln-analyzer-qwen-lora` |

## What 'good' looks like in cell 10

- Qwen's description follows the structured format — starts with "This function", contains a missing-check clause
- RoBERTa's top-1 matches the expected CWE for at least 2 of 3 test cases
- If RoBERTa top-1 is wrong but the correct CWE is in top-3, the system is usable — surface top-3 to the user

## If the descriptions look freeform and unstructured

The model didn't learn the output format — most likely the label-masking in cell 6 is off.  
Check `non_masked` in cell 6: if it's 0, labels are fully masked and the model learned nothing.  
If it's > 200, the prompt isn't being masked and the model is training on both prompt and response.

## Next: `pipeline/` implementation

Once cell 10 shows correct CWE classifications, both models are validated end-to-end.  
Next step is `pipeline/classifier.py` and `pipeline/code_analyzer.py` — wrapping these for the API.